# Particle-Filter Navigation Application

*Course 4 — Particle Filters, Part 3. A worked application of the [particle filter](16_Particle_Filter_SIS_Resampling.ipynb): **indoor navigation** without GPS, fusing dead-reckoning motion with noisy radio-beacon signal strength. Shows how the hard part of applying a PF is **modeling the pdfs**, not the algorithm.*

**Style:** every equation gets a plain-language paraphrase (→); extra intuition is flagged **→ Intuition**.

### 🧩 The Navigation Problem

- Three related but distinct problems:
  - **Target tracking** — estimate *another* object's state ("Where is *it*?").
  - **Navigation** — estimate *my own* state from local sensors ("Where am *I*?").
  - **Guidance** — compute a path to a goal ("How do I get there?" — a control problem).

- Most navigation **fuses** two kinds of measurement:
  - **Relative** (velocity/acceleration) from inertial sensors → dead-reckoning; drifts because errors and biases **integrate**.
  - **Absolute** (position fixes) → GPS outdoors; corrects the drift.

  → Relative-only is like running a filter open-loop (drift grows unbounded); absolute-only can be sparse or noisy. Fusing both gets the best of each.

- **→ Intuition:** an **INS** (inertial navigation system) predicts your motion; **GPS** (or, indoors, radio beacons) periodically anchors it. The filter blends the smooth-but-drifting prediction with the noisy-but-absolute fixes — exactly a predict/correct structure.

### 🧩 Indoor Navigation — No GPS

- GPS fails indoors (signal too weak, ~3 m accuracy too coarse). Absolute position instead comes from **radio beacons** at surveyed locations, via:
  - **Trilateration** — combine noisy **distances** to ≥3 known points (their uncertainty bands intersect at your position).
  - **Triangulation** — combine noisy **angles** to ≥3 known points.
  - Beacons use **UWB** time-of-flight or **RSSI** (received signal-strength indicator); WiFi/BLE/ZigBee also work.

  → Same geometric idea as GPS, but with local transmitters. With a **particle filter and a moving platform, even one distance or angle per step can suffice** — motion plus a map disambiguates position over time.

- **→ Intuition:** indoors, radio propagation is messy (walls, reflections, multipath) — the measurement model is **highly nonlinear and non-Gaussian**, which is precisely why a particle filter earns its keep here over an EKF.

### 🧩 The Motion (Process) Model

- The vehicle carries inertial sensors giving noisy **speed** $s_k$ and **heading** $\theta_k$. Position $[\xi_k,\eta_k]$ dead-reckons:

$$
\begin{bmatrix}\xi_{k+1}\\ \eta_{k+1}\end{bmatrix} = \begin{bmatrix}\xi_k\\ \eta_k\end{bmatrix} + \Delta t\begin{bmatrix} s_k\cos\theta_k\\ s_k\sin\theta_k\end{bmatrix}, \qquad s_k^m = s_k + w_k^s,\ \ \theta_k^m = \theta_k + w_k^\theta.
$$

  → Move in the heading direction by speed × time. The measured speed and heading each carry Gaussian noise $w_k^s\sim\mathcal N(0,\sigma_s^2)$, $w_k^\theta\sim\mathcal N(0,\sigma_\theta^2)$.

- **→ Intuition:** this is the particle **proposal** step — push each particle forward with the noisy motion command. Because the noise enters through $\cos$/$\sin$, the propagated distribution is **not** Gaussian, which is the crux of the modeling difficulty below.

### 🧩 The Measurement (RSSI Path-Loss) Model

- The measurement $z_k$ is the vector of **power losses** to the beacons. The deterministic part combines free-space loss with wall penetration; a **log-normal shadowing** term adds randomness:

$$
\bar{S} = (\text{loss at 1 m}) + 20\log_{10}(d) + N_w\times(\text{loss per wall}), \qquad \text{Loss} = \bar{S} + S,\ \ S\sim\mathcal{N}(0,\sigma_{\ln}^2),
$$

  where $d^2 = (\xi_k^{(i)}-x)^2 + (\eta_k^{(i)}-y)^2$ is the particle-to-beacon distance and $N_w$ counts intervening walls (from the map geometry).

- Because the random part $S$ is **additive Gaussian**, the measurement likelihood is Gaussian in the loss:

$$
f\big(z_k \mid x_k^{(i)}\big) = \mathcal{N}\big(\bar{S},\, \sigma_{\ln}^2\big).
$$

  → For each particle, compute its expected loss to every beacon (using distance **and** the number of walls the ray crosses), then score how close the *actual* measured loss is under a Gaussian. That score is the particle's likelihood weight.

- **→ Intuition:** the map (walls) makes the likelihood **spatially complex and multimodal** — two very different rooms can give the same signal strength. Only the particle representation captures that ambiguity honestly; a Gaussian filter would average incompatible hypotheses into a wrong single blob.

### 🧩 Setting Up the Filter — the Real Work

- To run the PF we need three things: the measurement likelihood $f(z_k\mid x_k^{(i)})$, the transition density $f(x_k^{(i)}\mid x_{k-1}^{(i)})$, and an importance density $q$.

- $f(z_k\mid x_k^{(i)})$ is the Gaussian RSSI likelihood above. $f(x_k^{(i)}\mid x_{k-1}^{(i)})$ is **hard** in closed form (noise through $\cos$/$\sin$); it is approximated as Gaussian two ways:
  - **Sigma-point method** — push sigma points through the motion map (unscented-transform statistics, as in [13](13_Sigma_Point_Unscented_Kalman_Filter.ipynb)).
  - **Coordinate-transformation method** — the polar-to-Cartesian debiasing of [11](11_Target_Tracking_IMM_AlphaBetaGamma.ipynb).

- Choosing $q(x_k^{(i)}\mid x_{k-1}^{(i)},z_k) = f(x_k^{(i)}\mid x_{k-1}^{(i)})$ (the prior) simplifies the weight to the **likelihood only**:

$$
\tilde{w}_k^{(i)} = f\big(z_k \mid x_k^{(i)}\big)\,\tilde{w}_{k-1}^{(i)}.
$$

  → Same convenient bootstrap choice as [16](16_Particle_Filter_SIS_Resampling.ipynb): propose from the motion model, weight by RSSI likelihood.

- **→ Intuition:** notice that all the effort went into **modeling the application's densities** — the PF algorithm itself (propose, weight, resample) is unchanged from the generic version. *"The real challenge is understanding and modeling the application, not implementing the particle filter."*

### 🧩 Implementation and Results

- The Octave implementation: initialize particles randomly inside the (L-shaped) room; each step, propose each particle forward with the noisy motion model, **constrain** particles to stay inside the outer walls, compute each one's expected RSSI to every beacon (counting wall crossings), weight by the Gaussian likelihood, normalize, estimate position as the weighted mean, and **resample** when $N_{\text{eff}}$ drops.

```octave
for cp = 1:Np                                  % for each particle
  P(:,cp) = P(:,cp) + [vxhat; vyhat] + chol(Sa,'lower')*randn(2,1); % propose
  P(:,cp) = max(0, min(W, P(:,cp)));           % constrain to room
  fz = 1;
  for b = 1:NB                                 % likelihood over beacons
    dist = norm(P(:,cp) - B(:,b));
    NW   = numWalls(P(:,cp), B(:,b), F, W);    % walls between particle & beacon
    fz   = fz * normpdf(z(b), L1 + 20*log10(dist) + LW*NW, sFade);
  end
  WW(cp) = WW(cp) * fz;                        % update weight
end
WW = WW / sum(WW);                             % normalize
% ... resample if 1/sum(WW.^2) < threshold ...
```

- **Result:** with more particles (10 → 20 → 100), initial and ongoing position estimates improve; all runs end with particles **clustered tightly around the true vehicle position**, even in the ambiguous L-shaped geometry with only two beacons.

- **→ Intuition:** the code that is *specific to the particle filter* is essentially unchanged from the generic algorithm; almost everything else is problem setup (room geometry, wall counting, RSSI model). The lesson generalizes: **applying a PF = modeling your pdfs well.**

### 🧩 Summary

- **Navigation** ("where am I?") fuses drifting **relative** (INS dead-reckoning) motion with **absolute** position fixes; indoors, GPS is replaced by **radio-beacon** trilateration/triangulation (UWB/RSSI).

- The **motion model** proposes particles forward with noisy speed/heading; the **RSSI path-loss model** (free-space + wall penetration + log-normal shadowing) gives a Gaussian **likelihood** per particle — but a **multimodal** posterior over the map, which is why a PF is used.

- The transition density (noise through $\cos$/$\sin$) is approximated via **sigma points** or **coordinate transformation**; the bootstrap choice $q=$ prior reduces the weight to the RSSI likelihood.

- The implementation is the generic PF (propose → weight → resample) plus problem-specific geometry; more particles give tighter estimates.

- **Takeaway for the whole specialization:** the filter algorithms are general and reusable — the craft is in **modeling the system, noises, and densities** for your specific application.

---
*Specialization notes complete. See [00 · Index](00_Index.ipynb) for the full map, or return to [Course 2 · Linear KF](07_Sequential_Probabilistic_Inference_Six_Steps.ipynb).*